# Construction Safety Vision — reproducible Colab training
One GPU run downloads the SHA-pinned dataset, trains YOLO11s, evaluates the untouched test split, exports ONNX, benchmarks it, and saves release-ready artifacts. Select **Runtime → Change runtime type → GPU** first.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.3.221 onnx==1.18.0 onnxslim==0.1.71 onnxruntime-gpu==1.23.2
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

In [ ]:
from pathlib import Path
import os, shutil, subprocess
REPO = Path('/content/Construction-Safety-Vision')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/YousefAl0taibi/Construction-Safety-Vision.git',str(REPO)],check=True)
os.chdir(REPO)
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

## Dataset
Atharion PPE v1.0-public has 791 train, 170 validation, and 169 scene-group-separated test images. Its mixed AGPL-3.0/CC BY 4.0 notice and attribution files must be retained. The ZIP hash is verified before extraction.

In [ ]:
import hashlib, urllib.request, zipfile
DATA_URL='https://github.com/atharion-team/ppe-dataset-v1/releases/download/v1.0/atharion-v1.0-public.zip'
EXPECTED_SHA256='f5f6f1966c4bdfff4151ee68fa0da21f446e1b788d15280e5f70028c69ec88c5'
archive=Path('/content/atharion-v1.0-public.zip')
if not archive.exists(): urllib.request.urlretrieve(DATA_URL,archive)
actual=hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual==EXPECTED_SHA256,f'Dataset hash mismatch: {actual}'
with zipfile.ZipFile(archive) as zf: zf.extractall('/content')
candidates=list(Path('/content').glob('atharion-v1.0-public/data.yaml'))
assert len(candidates)==1,candidates
DATA_YAML=candidates[0]
print('Dataset:',DATA_YAML)

In [ ]:
IMAGE_EXTENSIONS={'.jpg','.jpeg','.png','.webp'}
for split in ('train','valid','test'):
    images=[p for p in (DATA_YAML.parent/'images'/split).rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS]
    labels=list((DATA_YAML.parent/'labels'/split).glob('*.txt'))
    image_stems,label_stems={p.stem for p in images},{p.stem for p in labels}
    print(split,'images=',len(images),'labels=',len(labels))
    assert image_stems<=label_stems,f'{split}: images without labels: {sorted(image_stems-label_stems)[:5]}'
assert len(list((DATA_YAML.parent/'images'/'test').glob('*')))==169

In [ ]:
model=YOLO('yolo11s.pt')
model.train(data=str(DATA_YAML),epochs=80,imgsz=640,batch=-1,device=0,patience=15,workers=2,project='/content/runs',name='ppe-yolo11s',seed=42,deterministic=True,cache=False,amp=True,degrees=5,translate=0.1,scale=0.4,fliplr=0.5,mosaic=0.5,close_mosaic=10)

In [ ]:
import json, datetime
best_path=Path('/content/runs/ppe-yolo11s/weights/best.pt')
best=YOLO(best_path)
metrics=best.val(data=str(DATA_YAML),split='test',device=0,imgsz=640,plots=True,project='/content/eval',name='test')
names=metrics.names
report={'evaluatedAtUtc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'dataset':'atharion-v1.0-public','datasetSha256':EXPECTED_SHA256,'testImages':169,'model':'YOLO11s','imageSize':640,'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map),'perClass_mAP50_95':{names[i]:float(value) for i,value in enumerate(metrics.box.maps)},'gitCommit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'gpu':torch.cuda.get_device_name(0),'ultralytics':ultralytics.__version__}
Path('/content/eval/test/metrics.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))

In [ ]:
onnx_path=Path(best.export(format='onnx',imgsz=640,opset=17,simplify=True,dynamic=False))
shutil.copy2(onnx_path,REPO/'models'/'safety_detector.onnx')
subprocess.run(['python','scripts/benchmark.py','--model','models/safety_detector.onnx','--device','cuda','--runs','200','--output','reports/benchmark.json'],check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ARTIFACTS=Path('/content/drive/MyDrive/construction-safety/release-v0.1.0')
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source in [best_path,REPO/'models'/'safety_detector.onnx',Path('/content/eval/test/metrics.json'),REPO/'reports'/'benchmark.json']:
    shutil.copy2(source,ARTIFACTS/source.name)
for pattern in ('confusion_matrix*.png','PR_curve.png','P_curve.png','R_curve.png','F1_curve.png','results.csv'):
    for source in list(Path('/content/eval/test').glob(pattern))+list(Path('/content/runs/ppe-yolo11s').glob(pattern)):
        shutil.copy2(source,ARTIFACTS/source.name)
print('Release artifacts:',sorted(p.name for p in ARTIFACTS.iterdir()))

## Publish only after review
Download `release-v0.1.0`, commit `metrics.json` and `benchmark.json` into `reports/`, then create GitHub Release `v0.1.0` with `safety_detector.onnx`, `best.pt`, and the plots. Do not publish if the smoke test or held-out evaluation failed.